In [24]:
import pandas as pd

In [25]:
def load_and_prep(file_path, date_col, value_cols, new_col_names=None, delimiter=','):
    if file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path, sep=delimiter)
    df = df.rename(columns={date_col: 'Date'})
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

    cols_to_keep = ['Date'] + value_cols
    df = df[cols_to_keep].dropna(subset=['Date'])
    
    if new_col_names is not None:
        rename_dict = dict(zip(value_cols, new_col_names))
        df = df.rename(columns=rename_dict)
    
    df = df.set_index('Date')
    df = df.sort_index() 
    return df

In [ ]:
df_de10yt = load_and_prep('./Macro/DE10YT=RR.xlsx', 'Date', ['BidYld'], ['DE10YT_Yield'])
df_eur = load_and_prep('./Macro/EUR=.xlsx', 'Exchange Date', ['Bid', 'Refresh Rate'], ['EURUSD_Close', 'EURUSD_Volume'])
df_euribor = load_and_prep('./Macro/EURIBOR3M.csv', 'DATE', ['Euribor_3M'], delimiter=';')
df_ftmib = load_and_prep('./Macro/FTMIB.xlsx', 'Exchange Date', ['Close', 'Volume'], ['FTMIB_Close', 'FTMIB_Volume'])
df_gvz = load_and_prep('./Macro/GVZ.csv', 'DATE', ['GVZ'], ['GVZ_Close'])
df_it10yt = load_and_prep('./Macro/IT10YT=RR.xlsx', 'Name', ['ITALY GVT BMK BID YLD 10Y - RED. YIELD'], ['IT10YT_Yield'])
df_infl = load_and_prep('./Macro/ITCPANNL.xlsx', 'Name', ['IT INFLATION RATE NADJ'], ['IT_Inflation'])
df_itgdry = load_and_prep('./Macro/ITGDRY=ECI.xlsx', 'Name', ['IT GDP FINAL (%YOY) (CAL ADJ) SADJ'], ['IT_GDP'])
df_unemp = load_and_prep('./Macro/ITUNPTOTO.xlsx', 'Name', ['IT UNEMPLOYMENT RATE SADJ'], ['IT_Unemployment'])
df_oil = load_and_prep('./Macro/LCOc1.xlsx', 'Exchange Date', ['Close', 'Volume'], ['Brent_Close', 'Brent_Volume'])
df_ovx = load_and_prep('./Macro/OVX.csv', 'DATE', ['OVX'], ['OVX_Close'])
df_gas = load_and_prep('./Macro/TRNLTTFMc1.xlsx', 'Exchange Date', ['Close'], ['TTF_Gas_Close'])
df_vix = load_and_prep('./Macro/VIX.csv', 'DATE', ['CLOSE'], ['VIX_Close'], delimiter=';')
df_gold = load_and_prep('./Macro/XAUEUR=R.xlsx', 'Exchange Date', ['Bid'], ['Gold_Close'])

In [ ]:
dataframes = [df_de10yt, df_eur, df_euribor, df_ftmib, 
              df_gvz, df_it10yt, df_infl, df_itgdry, 
              df_unemp, df_oil, df_ovx, df_gas, 
              df_vix, df_gold]
df_macro = dataframes[0].join(dataframes[1:], how='outer')

#filling with forward fill for monthly data
df_macro = df_macro.ffill()
df_macro = df_macro.dropna()
df_macro = df_macro.reset_index()

df_macro.to_csv('./Processed/Macro.csv', index=False)